# Plotting Training Metrics

Track **loss** and **accuracy** over epochs with matplotlib.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic classification data for CPU-only tutorials."""
    def __init__(self, n_samples=256, n_features=8, n_classes=3, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

    def tensors(self):
        return self.X, self.y

class SimpleMLP(nn.Module):
    def __init__(self, in_dim=8, hidden=16, n_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        return self.net(x)


## 1. Simulate training history

In [ ]:
epochs = list(range(1, 16))
train_loss = [1.2 * np.exp(-0.25 * e) + 0.05 + np.random.rand() * 0.03 for e in epochs]
val_loss = [1.1 * np.exp(-0.2 * e) + 0.12 + np.random.rand() * 0.05 for e in epochs]
train_acc = [1 - l / 1.3 for l in train_loss]
val_acc = [1 - l / 1.4 for l in val_loss]

## 2. Loss curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, train_loss, 'b-o', label='train loss', lw=2)
ax.plot(epochs, val_loss, 'r-o', label='val loss', lw=2)
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend()
ax.set_title('Training & validation loss')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Accuracy curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, train_acc, 'b-o', label='train acc', lw=2)
ax.plot(epochs, val_acc, 'r-o', label='val acc', lw=2)
ax.set_xlabel('epoch'); ax.set_ylabel('accuracy'); ax.legend()
ax.set_title('Training & validation accuracy')
plt.tight_layout(); plt.show()

## 4. Dual-axis plot

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
ax1.plot(epochs, train_loss, 'b-', label='loss')
ax2.plot(epochs, train_acc, 'g--', label='acc')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss', color='b')
ax2.set_ylabel('accuracy', color='g')
ax1.set_title('Loss and accuracy on shared epoch axis')
plt.tight_layout(); plt.show()

## 5. Live metrics from short training run

In [ ]:
gen = DummyDataGenerator(n_samples=200, n_features=8, n_classes=3)
X, y = gen.tensors()
model = SimpleMLP()
opt = torch.optim.Adam(model.parameters(), lr=0.02)
live_loss, live_acc = [], []
for ep in range(20):
    opt.zero_grad()
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward(); opt.step()
    live_loss.append(loss.item())
    live_acc.append((logits.argmax(1) == y).float().mean().item())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(live_loss, color='steelblue'); axes[0].set_title('Live loss')
axes[1].plot(live_acc, color='seagreen'); axes[1].set_title('Live accuracy')
plt.tight_layout(); plt.show()